In [ ]:
"""
=============================================================================
GENERALIZED MoE ENTROPY ANALYSIS FRAMEWORK
=============================================================================

HOW TO USE THIS FRAMEWORK FOR YOUR SPECIFIC MODEL:
1. Standard Hugging Face Models: If your model is on the HF Hub (e.g., Qwen, Mixtral, OLMoE),
   you usually only need to change the `model_id` in the execution block at the bottom.
   The `GenericHFMoEBackend` will automatically try to find the correct layer norms
   and router shapes.

2. Custom/Local Models: If you are using a heavily customized architecture where
   the generic backend fails (e.g., it can't find the final layer norm):
   - Create a new class that inherits from `ArchitectureBackend`.
   - Implement your own `load_model()` and `extract_entropy_data()` methods.
   - Look at `GenericHFMoEBackend` as a template for how to project hidden states
     through the LM head to get prediction probabilities.

HOW TO CREATE SPECIFIC VISUALIZATIONS:
The `extract_entropy_data` function returns a standardized Pandas DataFrame with
the following columns:
    - 'layer' (int): The depth of the model.
    - 'position' (int): The sequence index of the token.
    - 'token' (str): The actual text word/subword.
    - 'prediction_entropy' (float): The uncertainty of the LM head at that layer.
    - 'router_entropy' (float): The uncertainty of the MoE routing at that layer.

To create a new visualization:
1. Add a new `@staticmethod` to the `Visualizer` class.
2. Accept the DataFrame (`df`) as an argument.
3. Use Pandas to group, pivot, or filter the data (e.g., `df.pivot(...)` for heatmaps).
4. Use Matplotlib or Seaborn to plot the specific metrics you care about.
(See `plot_entropy_heatmap` below for a concrete example!)
=============================================================================
"""

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from abc import ABC, abstractmethod

# ==========================================
# 1. Entropy Computation Module
# ==========================================
class EntropyCalculator:
    @staticmethod
    def compute_shannon_entropy(probabilities):
        eps = 1e-9
        probs = torch.clamp(probabilities, min=eps)
        entropy = -torch.sum(probs * torch.log2(probs), dim=-1)
        return entropy.detach().cpu().float().numpy()

# ==========================================
# 2. Prompt / Dataset Loading Module
# ==========================================
class DatasetLoader:
    """
    INSTRUCTION: HOW TO LOAD SPECIFIC DATASETS

    1. Standard Hugging Face Hub Datasets:
       - Find a dataset on huggingface.co/datasets (e.g., "allenai/c4")
       - Set dataset_path="allenai/c4"
       - Set dataset_config="en" (if required by the dataset, otherwise None)
       - Set text_column="text" (or whatever column holds the text in that dataset)

    2. Local CSV or JSON Files:
       - Set dataset_path="csv" (or "json")
       - Set dataset_config=None
       - You will pass data_files={"test": "path/to/your/local_file.csv"} when calling load_sample_text().
    """
    def __init__(self, dataset_path="Salesforce/wikitext", dataset_config="wikitext-103-raw-v1", text_column="text"):
        self.dataset_path = dataset_path
        self.dataset_config = dataset_config
        self.text_column = text_column

    def load_sample_text(self, split="test", sample_index=10, data_files=None):
        print(f"Fetching {self.dataset_path} dataset...")

        # Load from HF Hub or Local File
        if data_files:
            dataset = load_dataset(self.dataset_path, data_files=data_files, split=split)
        elif self.dataset_config:
            dataset = load_dataset(self.dataset_path, self.dataset_config, split=split)
        else:
            dataset = load_dataset(self.dataset_path, split=split)

        # Extract the specific text column and filter out empty strings
        try:
            texts = [t for t in dataset[self.text_column] if isinstance(t, str) and len(t.strip()) > 50]
        except KeyError:
            raise KeyError(f"Column '{self.text_column}' not found in dataset. Available columns: {dataset.column_names}")

        if sample_index >= len(texts):
            raise IndexError(f"Sample index {sample_index} out of range. Only {len(texts)} valid samples found.")

        return texts[sample_index]

# ==========================================
# 3. Model & Architecture-Specific Extraction
# ==========================================
class ArchitectureBackend(ABC):
    @abstractmethod
    def load_model(self): pass

    @abstractmethod
    def extract_entropy_data(self, text): pass

class GenericHFMoEBackend(ArchitectureBackend):
    def __init__(self, model_id):
        self.model_id = model_id
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = None
        self.tokenizer = None

    def _get_layer_norm(self):
        """
        INSTRUCTION: If your model crashes with an AttributeError here, it means
        your model's final normalization layer has a custom name.
        Print(self.model) to find it, and add the path (e.g., self.model.custom_norm) here.
        """
        if hasattr(self.model, 'model') and hasattr(self.model.model, 'norm'):
            return self.model.model.norm
        elif hasattr(self.model, 'ln_f'):
            return self.model.ln_f
        elif hasattr(self.model, 'norm'):
            return self.model.norm
        else:
            raise AttributeError(f"Could not dynamically find the final layer norm for {self.model_id}.")

    def load_model(self):
        print(f"Loading {self.model_id} on {self.device}...")
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_id,
            output_router_logits=True,
            output_hidden_states=True,
            trust_remote_code=True,
            torch_dtype=torch.bfloat16
        ).to(self.device)
        self.model.eval()

    def extract_entropy_data(self, text, max_tokens=32):
        print("Running forward pass and extracting layer-wise states...")
        inputs = self.tokenizer(text, return_tensors="pt", max_length=max_tokens, truncation=True).to(self.device)
        tokens = self.tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
        seq_len = len(tokens)

        with torch.no_grad():
            outputs = self.model(**inputs)

        data = []
        hidden_states = outputs.hidden_states
        router_logits = outputs.router_logits if hasattr(outputs, 'router_logits') else []
        num_layers = len(hidden_states) - 1

        layer_norm = self._get_layer_norm()
        lm_head = self.model.lm_head

        for layer_idx in range(num_layers):
            current_hidden_state = hidden_states[layer_idx + 1]
            normalized_state = layer_norm(current_hidden_state)
            layer_vocab_logits = lm_head(normalized_state)
            layer_vocab_probs = torch.softmax(layer_vocab_logits, dim=-1)
            pred_entropies = EntropyCalculator.compute_shannon_entropy(layer_vocab_probs[0])

            if layer_idx < len(router_logits) and router_logits[layer_idx] is not None:
                layer_router_logits = router_logits[layer_idx]
                if len(layer_router_logits.shape) == 3:
                    router_probs = torch.softmax(layer_router_logits[0], dim=-1)
                else:
                    router_probs = torch.softmax(layer_router_logits, dim=-1)
                    router_probs = router_probs.view(seq_len, -1)
                router_entropies = EntropyCalculator.compute_shannon_entropy(router_probs)
            else:
                router_entropies = [0.0] * seq_len

            for pos, token in enumerate(tokens):
                data.append({
                    'layer': layer_idx,
                    'position': pos,
                    'token': token,
                    'prediction_entropy': pred_entropies[pos],
                    'router_entropy': router_entropies[pos]
                })

        return pd.DataFrame(data)

# ==========================================
# 4. Plotting & Visualization Module
# ==========================================
class Visualizer:
    @staticmethod
    def plot_average_moe_entropy(df, save_path="average_moe_entropy.png"):
        df_avg = df.groupby('layer')[['prediction_entropy', 'router_entropy']].mean().reset_index()
        sns.set_theme(style="whitegrid")
        fig, ax = plt.subplots(figsize=(10, 6))
        sns.lineplot(data=df_avg, x='layer', y='prediction_entropy', label='Avg Prediction', marker='o', color='#e74c3c', ax=ax)
        if 'router_entropy' in df_avg.columns:
            sns.lineplot(data=df_avg, x='layer', y='router_entropy', label='Avg MoE Router', marker='s', color='#2ecc71', ax=ax)
        ax.set_title("Average Entropy Across Physical Model Depth", pad=15, fontweight='bold')
        ax.set_xlabel("Model Depth (Layers)")
        ax.set_ylabel("Shannon Entropy (Bits)")
        plt.tight_layout()
        plt.savefig(save_path, dpi=300)
        plt.close()

    @staticmethod
    def plot_single_token_journey(df, target_position, save_path="single_token_entropy.png"):
        df_single = df[df['position'] == target_position].copy()
        if df_single.empty: return
        token_word = df_single.iloc[0]['token']

        sns.set_theme(style="whitegrid")
        fig, ax = plt.subplots(figsize=(10, 6))
        sns.lineplot(data=df_single, x='layer', y='prediction_entropy', label='Prediction', marker='o', color='#3498db', ax=ax)
        if 'router_entropy' in df_single.columns:
            sns.lineplot(data=df_single, x='layer', y='router_entropy', label='MoE Router', marker='s', color='#f39c12', ax=ax)
        ax.set_title(f"Resolution Journey for Token '{token_word}' (Pos: {target_position})", pad=15, fontweight='bold')
        ax.set_xlabel("Model Depth (Layers)")
        ax.set_ylabel("Shannon Entropy (Bits)")
        plt.tight_layout()
        plt.savefig(save_path, dpi=300)
        plt.close()

    # --- INSTRUCTION: HOW TO ADD CUSTOM VISUALIZATIONS ---
    # Example: A user wants to see a heatmap of prediction entropy for all tokens across all layers.
    # 1. Take the dataframe 'df'
    # 2. Reshape it using pd.pivot
    # 3. Plot using sns.heatmap
    @staticmethod
    def plot_entropy_heatmap(df, save_path="entropy_heatmap.png"):
        """CUSTOM VISUALIZATION EXAMPLE: Heatmap of prediction uncertainty."""
        # Clean token labels (sometimes they have special characters from tokenizers like 'Ġ')
        clean_tokens = [t.replace('Ġ', '').replace(' ', '') for t in df['token'].unique()]

        # Pivot the dataframe: Rows = Layers, Columns = Tokens, Values = Prediction Entropy
        pivot_df = df.pivot(index="layer", columns="position", values="prediction_entropy")

        plt.figure(figsize=(14, 8))
        sns.heatmap(pivot_df, cmap="YlGnBu", xticklabels=clean_tokens)

        plt.title("Prediction Entropy Heatmap (Sequence vs. Layers)", pad=15, fontweight='bold')
        plt.xlabel("Input Sequence (Tokens)")
        plt.ylabel("Transformer Layer")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig(save_path, dpi=300)
        plt.close()

# ==========================================
# 5. Pipeline Coordinator
# ==========================================
def run_entropy_pipeline(backend: ArchitectureBackend, text_input: str):
    print("--- Starting Entropy Analysis Pipeline ---")

    backend.load_model()
    df_results = backend.extract_entropy_data(text_input)

    # Generate visual artifacts
    Visualizer.plot_average_moe_entropy(df_results, "macro_average_entropy.png")

    target_pos = len(df_results['position'].unique()) - 2
    Visualizer.plot_single_token_journey(df_results, target_pos, f"token_{target_pos}_entropy.png")

    # Call the new custom visualization
    Visualizer.plot_entropy_heatmap(df_results, "custom_heatmap.png")

    df_results.to_csv("pipeline_results.csv", index=False)
    print("--- Pipeline Complete. Data saved to pipeline_results.csv ---")



In [ ]:
if __name__ == "__main__":
    # ---------------------------------------------------------
    # OPTION A: Using the default Wikitext dataset
    # ---------------------------------------------------------
    # loader = DatasetLoader()
    # sample_text = loader.load_sample_text(sample_index=10)


    # ---------------------------------------------------------
    # OPTION B: Using a different Hugging Face dataset (e.g., OpenOrca)
    # OpenOrca uses the column 'question' instead of 'text'
    # ---------------------------------------------------------
    # loader = DatasetLoader(
    #     dataset_path="Open-Orca/OpenOrca",
    #     dataset_config=None,
    #     text_column="question"
    # )
    # # Note: OpenOrca only has a 'train' split by default
    # sample_text = loader.load_sample_text(split="train", sample_index=5)


    # ---------------------------------------------------------
    # OPTION C: Using a local CSV file
    # Assuming your CSV has a column called 'prompt'
    # ---------------------------------------------------------
    loader = DatasetLoader()
    sample_text = loader.load_sample_text()

    # STEP 2: DEFINE YOUR MODEL
    # To test a new Hugging Face model, simply paste its repository ID here.
    # Examples:
    # "allenai/OLMoE-1B-7B-0924"
    # "Qwen/Qwen1.5-MoE-A2.7B"
    # "mistralai/Mixtral-8x7B-v0.1" (Warning: Requires heavy GPU memory!)
    CUSTOM_MODEL_ID = "allenai/OLMoE-1B-7B-0924"

    moe_backend = GenericHFMoEBackend(model_id=CUSTOM_MODEL_ID)

    # STEP 3: RUN THE PIPELINE
    # The pipeline will handle extraction, call the Visualizer methods, and save the CSV.
    run_entropy_pipeline(moe_backend, sample_text)

Fetching Salesforce/wikitext dataset...
--- Starting Entropy Analysis Pipeline ---
Loading allenai/OLMoE-1B-7B-0924 on cuda...


Loading weights: 100%|██████████| 179/179 [00:02<00:00, 82.83it/s]


Running forward pass and extracting layer-wise states...
--- Pipeline Complete. Data saved to pipeline_results.csv ---
